# CSI 300 parallel minute-cache generator

Read raw market pickle files, build market partitions, merge one complete `basket_tick_v03` cache per trading day, and optionally remove the partitions after validation.


In [7]:
from pathlib import Path
import sys

_root_candidates = [Path.cwd().resolve(), Path.cwd().resolve() / "Stock-Index-Fitting"]
PROJECT_ROOT = next(
    (candidate for candidate in _root_candidates if (candidate / "utils" / "cache_generator.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Cannot locate Stock-Index-Fitting from the current working directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import display
from xtquant import xtdata

from utils.cache_generator import (
    CSI300_INDEX_CODE,
    generate_csi300_caches,
    merge_partition_caches_for_range,
    preview_generation_plan,
    resolve_trading_dates,
)


## Configuration

`START_DATE` and `END_DATE` control both partition generation and complete-cache merging. Non-trading endpoints are moved to the closest previous SH trading day through the XtQuant calendar.


In [ ]:
INDEX_CODE = CSI300_INDEX_CODE  # Fixed to CSI 300: 000300
START_DATE = "20260703"
END_DATE = "20260726"
XT_PORT = 58610
MAX_WORKERS = 2
FORCE_REBUILD = False

# Delete validated market partition pickle files after a complete cache exists.
DELETE_PARTITION_CACHES = True

# Rebuild an already valid complete cache during the merge phase.
OVERWRITE_COMPLETE_CACHE = False

SOURCE_TICK_ROOT = Path(r"\\192.168.1.138\康曼德共享\高频行情迅投\ticks")
WEIGHTS_DIR = PROJECT_ROOT / "data" / "weights_projection"
CACHE_DIR = PROJECT_ROOT / "data" / INDEX_CODE / "_tick_cache_correlation_v03"

print("Project root:", PROJECT_ROOT)
print("Source tick root:", SOURCE_TICK_ROOT)
print("Weights directory:", WEIGHTS_DIR)
print("Cache directory:", CACHE_DIR)
print("Workers:", MAX_WORKERS)
print("Force rebuild:", FORCE_REBUILD)
print("Delete partition caches:", DELETE_PARTITION_CACHES)


Project root: E:\Codex\系统\Stock-Index-Fitting
Source tick root: \\192.168.1.138\康曼德共享\高频行情迅投\ticks
Weights directory: E:\Codex\系统\Stock-Index-Fitting\data\weights_projection
Cache directory: E:\Codex\系统\Stock-Index-Fitting\data\000300\_tick_cache_correlation_v03
Workers: 2
Force rebuild: False
Delete partition caches: True


## Resolve trading dates


In [9]:
xtdata.reconnect(port=XT_PORT)
date_resolution = resolve_trading_dates(
    START_DATE,
    END_DATE,
    xtdata_client=xtdata,
)
trade_dates = list(date_resolution.trade_dates)

print("Configured range:", date_resolution.configured_start_date, date_resolution.configured_end_date)
print("Adjusted range:  ", date_resolution.adjusted_start_date, date_resolution.adjusted_end_date)
print(f"Trading dates ({len(trade_dates)}):", trade_dates)


***** xtdata连接成功 2026-07-27 10:20:59*****
服务信息: {'tag': 'qmt_research', 'version': '1.0'}
服务地址: 127.0.0.1:58610
数据路径: E:\迅投极速交易终端睿智融科版\datadir
设置xtdata.enable_hello = False可隐藏此消息

Configured range: 20260703 20260711
Adjusted range:   20260703 20260710
Trading dates (6): ['20260703', '20260706', '20260707', '20260708', '20260709', '20260710']


## Preview inputs

This metadata-only check does not deserialize the large source pickle files.


In [10]:
plan = preview_generation_plan(
    trade_dates,
    weights_dir=WEIGHTS_DIR,
    source_tick_root=SOURCE_TICK_ROOT,
    cache_dir=CACHE_DIR,
)
display(plan)

if "status" in plan.columns and plan["status"].eq("plan_error").any():
    raise RuntimeError("The generation plan contains weight/source mapping errors.")
if not plan["source_exists"].fillna(False).all():
    missing = plan.loc[~plan["source_exists"].fillna(False), "source_path"].tolist()
    raise FileNotFoundError(f"Missing source pickle files: {missing}")


,trade_date,weight_file,component_count,market,market_stock_count,source_path,source_exists,source_gb,final_cache_path,final_cache_exists
0,20260703,沪深300_样本权重_20260630.csv,300,sh_kcb,20,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SH\2026\07\...,True,0.733,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,False
1,20260703,沪深300_样本权重_20260630.csv,300,sh_zb,169,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SH\2026\07\...,True,2.105,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,False
2,20260703,沪深300_样本权重_20260630.csv,300,sz_cyb,34,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SZ\2026\07\...,True,1.595,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,False
3,20260703,沪深300_样本权重_20260630.csv,300,sz_zb,77,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SZ\2026\07\...,True,1.741,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,False
4,20260706,沪深300_样本权重_20260630.csv,300,sh_kcb,20,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SH\2026\07\...,True,0.732,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,False
5,20260706,沪深300_样本权重_20260630.csv,300,sh_zb,169,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SH\2026\07\...,True,2.096,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,False
6,20260706,沪深300_样本权重_20260630.csv,300,sz_cyb,34,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SZ\2026\07\...,True,1.578,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,False
7,20260706,沪深300_样本权重_20260630.csv,300,sz_zb,77,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SZ\2026\07\...,True,1.741,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,False
8,20260707,沪深300_样本权重_20260630.csv,300,sh_kcb,20,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SH\2026\07\...,True,0.718,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,False
9,20260707,沪深300_样本权重_20260630.csv,300,sh_zb,169,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SH\2026\07\...,True,2.074,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,False


## Generate market partitions and complete caches

Dates are processed sequentially. Physical market files within one date are handled by the configured process pool.


In [11]:
generated_cache_paths = generate_csi300_caches(
    trade_dates,
    weights_dir=WEIGHTS_DIR,
    source_tick_root=SOURCE_TICK_ROOT,
    cache_dir=CACHE_DIR,
    max_workers=MAX_WORKERS,
    force_rebuild=FORCE_REBUILD,
)

print(f"Complete caches available after generation: {len(generated_cache_paths)}")
for cache_path in generated_cache_paths:
    print(" ", cache_path)


[DATE 1/6] 20260703: preparing CSI 300 cache with max_workers=2
  weights=沪深300_样本权重_20260630.csv, components=300
  [partition:sh_kcb] built (141.8s)
  [partition:sh_zb] built (183.5s)
  [partition:sz_cyb] built (48.4s)
  [partition:sz_zb] built (36.1s)
  [final:all] built: basket_minute_wide_20260703_bfa50f229284.pkl
[DATE 2/6] 20260706: preparing CSI 300 cache with max_workers=2
  weights=沪深300_样本权重_20260630.csv, components=300
  [partition:sh_kcb] built (142.2s)
  [partition:sh_zb] built (306.9s)
  [partition:sz_cyb] built (169.4s)
  [partition:sz_zb] built (37.5s)
  [final:all] built: basket_minute_wide_20260706_b0ca6a949f2e.pkl
[DATE 3/6] 20260707: preparing CSI 300 cache with max_workers=2
  weights=沪深300_样本权重_20260630.csv, components=300
  [partition:sh_kcb] built (22.8s)
  [partition:sh_zb] built (65.7s)
  [partition:sz_cyb] built (46.8s)
  [partition:sz_zb] built (32.7s)
  [final:all] built: basket_minute_wide_20260707_14ca0077a818.pkl
[DATE 4/6] 20260708: preparing CSI 300 ca

## Merge, validate, and clean partitions

The merge range uses the same `START_DATE` and `END_DATE`. Partition files are deleted only after the complete cache passes schema, date, stock-universe, missing-stock, and minute-row validation.


In [12]:
complete_cache_paths = merge_partition_caches_for_range(
    START_DATE,
    END_DATE,
    xtdata_client=xtdata,
    weights_dir=WEIGHTS_DIR,
    source_tick_root=SOURCE_TICK_ROOT,
    cache_dir=CACHE_DIR,
    delete_partition_caches=DELETE_PARTITION_CACHES,
    overwrite_complete_cache=OVERWRITE_COMPLETE_CACHE,
)

print(f"Validated complete caches: {len(complete_cache_paths)}")
for cache_path in complete_cache_paths:
    print(" ", cache_path)


Merge trading dates (6): 20260703..20260710
[MERGE 1/6] 20260703
  [20260703] complete cache hit: basket_minute_wide_20260703_bfa50f229284.pkl
  [20260703] deleted partition: minute_partition_20260703_sh_kcb_df20c975af60.pkl
  [20260703] deleted partition: minute_partition_20260703_sh_zb_18ead6d8413f.pkl
  [20260703] deleted partition: minute_partition_20260703_sz_cyb_50c13bf1dc54.pkl
  [20260703] deleted partition: minute_partition_20260703_sz_zb_860162368b39.pkl
[MERGE 2/6] 20260706
  [20260706] complete cache hit: basket_minute_wide_20260706_b0ca6a949f2e.pkl
  [20260706] deleted partition: minute_partition_20260706_sh_kcb_c6651ed22dcc.pkl
  [20260706] deleted partition: minute_partition_20260706_sh_zb_fefe39aace3a.pkl
  [20260706] deleted partition: minute_partition_20260706_sz_cyb_b3111fbc7171.pkl
  [20260706] deleted partition: minute_partition_20260706_sz_zb_0019c2da1786.pkl
[MERGE 3/6] 20260707
  [20260707] complete cache hit: basket_minute_wide_20260707_14ca0077a818.pkl
  [2026